# Manuscript 1 — NeuroMap EML

Анализ модели `NeuroMapManuscriptEML`, обученной на том же датасете, что и `e1_train.py`.

Запускайте ноутбук с **корня репозитория** (`neuromap_sync`), чтобы импорты `neuromaps` и `systems` работали; либо добавьте корень в `PYTHONPATH`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "neuromaps").is_dir():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError("Не найден корень репозитория (папка neuromaps). Запустите из корня neuromap_sync.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from neuromaps import NeuroMapManuscriptEML
from systems.vdp_mod1 import vdp_mod1_rk4

CKPT = ROOT / "experiments/manuscript_1/checkpoints_eml/model.ckpt"
model = NeuroMapManuscriptEML.load(str(CKPT))
print("loaded", CKPT, "n_leaves=", model.n_leaves, "dt=", model.dt)

In [ ]:
hist = model.training_history
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(np.arange(len(hist["train_loss"])), hist["train_loss"], label="train (L1)", lw=1.5, alpha=0.85)
ax.plot(np.arange(len(hist["val_loss"])), hist["val_loss"], label="val (L1)", lw=1.5, alpha=0.85)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, ls="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def rk4_reference_traj(u0, p, dt, n_steps):
    u = np.asarray(u0, dtype=np.float64).ravel()
    p = np.asarray(p, dtype=np.float64).ravel()
    traj = [u.copy()]
    for _ in range(n_steps):
        u = vdp_mod1_rk4(u, p, dt)
        traj.append(u.copy())
    return np.asarray(traj)


DT = 0.01
N_STEPS = 5000
u0 = np.array([[5.0, 15.0]])
p = np.array([[-0.5, 0.05]])

nm_traj = model.simulate(u0, p, n_steps=N_STEPS, verbose=False, divergence_threshold=1e6)
ode_traj = rk4_reference_traj(u0.ravel(), p.ravel(), DT, N_STEPS)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
if nm_traj is not None:
    ax.plot(nm_traj[:, 0], nm_traj[:, 1], lw=0.8, alpha=0.9, label="NeuroMap EML")
else:
    print("NeuroMap: траектория разошлась")
ax.plot(ode_traj[:, 0], ode_traj[:, 1], lw=0.8, alpha=0.6, label="RK4 (vdp_mod1)")
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("Фазовый портрет: сравнение с эталонным RK4")
ax.legend()
ax.grid(True, ls="--", alpha=0.3)
ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn.functional as F

fig, axes = plt.subplots(1, model.n_var, figsize=(5 * model.n_var, 3.5))
if model.n_var == 1:
    axes = [axes]
for k in range(model.n_var):
    w = F.softmax(model.eml_trees[k].leaf_logits, dim=-1).detach().cpu().numpy()
    labels = ["1", r"$u_1$", r"$u_2$", r"$p_1$", r"$p_2$"][: w.shape[1]]
    im = axes[k].imshow(w, aspect="auto", cmap="viridis")
    axes[k].set_title(f"Листья → терминалы, координата {k+1}")
    axes[k].set_ylabel("leaf index")
    axes[k].set_xticks(range(len(labels)))
    axes[k].set_xticklabels(labels, rotation=30)
    plt.colorbar(im, ax=axes[k], fraction=0.046)
plt.tight_layout()
plt.show()